# Load minIO bronze to MinIO silver (Incremental)

### Install python dotenv for get the environment variables

In [3]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install delta-spark==3.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 5.4 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


## Imports libs, files and configure the absolute path

In [5]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession, functions
from pyspark.sql.functions import date_format
import logging
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from delta.tables import DeltaTable

# Import for get the environment variables 
from dotenv import load_dotenv
import os
import sys
sys.path.append(os.path.abspath("../../")) # Used to reconfigure the absolut path. In this case, setting the absolut path to 2 folders back (notebooks/...) 
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder

## Load environment variables

In [6]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")

## Spark configurations

In [7]:
def configure_spark():
    conf = SparkConf()
    
    conf.setAppName("Incremental transform from MinIO bronze to MinIO silver") # Spark application name, Usefull for logs
    # Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
    # add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
    conf.set("spark.jars.packages", 
             "org.apache.hadoop:hadoop-aws:3.3.4,"
             "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
             "org.postgresql:postgresql:42.7.2,"
             "io.delta:delta-spark_2.12:3.1.0" )
    conf.set("spark.master", "spark://spark-master:7077") # set the Spark container to be distributed among the workers
    conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
    conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
    conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
    conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
    conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
    conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
    conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
    conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external
    
    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    return spark

## Configuration the Logging and log the startup

In [8]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

## Creating a method to process each table when is called on method main

In [9]:
def process_table(spark, table_name, query, output_table_path):

    try:
        
        # Logging the processing
        logging.info(f"processing table {table_name}")
        
        try:

            df_silver = spark.read.format("delta").load(output_table_path)

            # Verify if the table have rows before the collect function
            if df_silver.count() > 0:
                
                # Getting max date value from minIO silver in the modifieddate column. limit at 1 result and get this result on 1º row at max_modifieddate column
                df_max_modifieddate_silver = df_silver.select(functions.max("modifieddate").alias("max_modifieddate")) \
                .limit(1).collect()[0]["max_modifieddate"]
                
                # If the table exists but is empty, or if any row has an empty “modifiedate” column (return None)
                if df_max_modifieddate_silver is None:
                    
                    df_max_modifieddate_silver = "1900-01-01 00:00:00"

            else:
                df_max_modifieddate_silver = "1900-01-01 00:00:00"
                
        except Exception:
            
            logging.info(f"Silver table for {table_name} not found. Starting initial load.")
            
            # If the path does not exist in MinIO (First run)
            df_max_modifieddate_silver = "1900-01-01 00:00:00"
        
        
         #Transforming data from the bronze layer where the “modifieddate” column is more recent than the “modifieddate” column in the silver layer
        query_update_data_to_silver = spark.sql(f"""
            select * from ({query}) as subquery
            where modifieddate > '{df_max_modifieddate_silver}'
            """)
    
        # Number of rows returns from query to update, if exists
        rows_to_update = query_update_data_to_silver.count()
        
        if rows_to_update == 0:
            # Logging if get no rows to update in minio bronze
            logging.info(f"No new data to process for table {table_name}")
    
        else:
            # Logging number of rows to update
            logging.info(f"Number of new rows to update for table {table_name}: {rows_to_update}")
            
            # Adding a new column date related the load data
            df_with_last_update = func_file.add_data_last_update(query_update_data_to_silver)
    
            # modifing dataframe to add a new column "month_key" to create a partition on the minIO silver based on modifieddate column
            df_with_month_partition = df_with_last_update.withColumn("month_key", date_format(df_with_last_update["modifieddate"], "yyyy-MM"))

            # Verify if the table path exists
            if not DeltaTable.isDeltaTable(spark, output_table_path):
                # If the table does not exist, create it.
                logging.info(f"First load. Creating table {table_name}...")
                df_with_month_partition.write.format("delta").mode("overwrite").partitionBy("month_key").save(output_table_path)
                
            else:
                # Updating the dataframe on minIO silver
                logging.info(f"Updating table {table_name}...")
                
                # If the table exists, do a merge
                # Instance of the target Silver Delta Table
                target_table = DeltaTable.forPath(spark, output_table_path)

                # Starting with the silver tier, you don't need to create the hash because it was already created in the bronze tier.
                # Therefore, if the tier doesn't exist, all new rows will be added; if it does exist, a comparison will be made
                # with the ‘row_hash’ column of each row. If it matches, the entire row will be updated; 
                # if it doesn't match any row, it will be inserted.
                target_table.alias("target") \
                    .merge(
                        df_with_month_partition.alias("updates"),
                        "target.row_hash = updates.row_hash" 
                    ) \
                    .whenMatchedUpdateAll() \
                    .whenNotMatchedInsertAll() \
                    .execute()
                
            # Logging the sucessfully process
            logging.info(f"Table {table_name} Sucessfully updated and saved in MinIO silver on: {output_table_path}")
            
    except Exception as e:
        # Logging the Error
         logging.error(f"Error processing table {table_name}: {str(e)}")

In [10]:
if __name__ == "__main__":
    
    # Logging the Start process from ingestion
    logging.info("Starting incrmental transform from MinIO bronze to MinIO silver...")

    spark = configure_spark()

    # bronze path
    bronze_path = config_file.data_lakehouse_path["bronze"]
    # silver path
    silver_path = config_file.data_lakehouse_path["silver"]

    queries_tables = config_file.queries_silver

    # Creating a ThreadPool for divide all jobs among the workers and execute in parallel
    with ThreadPoolExecutor(max_workers=8) as executor:
        
        # Creating a list to Add all jobs into it
        futures = []

        for table_name in config_file.queries_silver.keys():
            
            # bronze table path
            bronze_table_path = f"{bronze_path}bronze_{table_name}"
            
            # silver table path
            silver_table_path = f"{silver_path}silver_{table_name}"
    
            query = func_file.get_query(table_name, queries_tables, bronze_path)

            # Instead of calling the function to execute it, call the function by passing it to the executor
            futures.append(executor.submit(process_table, spark, table_name, query, silver_table_path))

       
        for future in as_completed(futures):
            try:
                # Where the all jobs is executing by the executors created with ThreadPoolExecutor
                future.result()

            except Execption as e:
                logging.error(f"Error in one of parallel taks: {str(e)}")
            

    # Logging the Incremental ingestion
    logging.info(f"Incremental ingestion to silver layer completed!")
    

2026-06-03 22:03:29,467 - INFO - Starting incrmental transform from MinIO bronze to MinIO silver...
2026-06-03 22:03:34,389 - INFO - processing table sales_countryregioncurrency
2026-06-03 22:03:34,392 - INFO - processing table sales_creditcard
2026-06-03 22:03:34,393 - INFO - processing table sales_currency
2026-06-03 22:03:34,396 - INFO - processing table sales_currencyrate
2026-06-03 22:03:34,402 - INFO - processing table sales_salesorderdetail
2026-06-03 22:03:34,399 - INFO - processing table sales_personcreditcard
2026-06-03 22:03:34,397 - INFO - processing table sales_customer
2026-06-03 22:03:34,413 - INFO - processing table sales_salesorderheader
2026-06-03 22:03:40,048 - INFO - Silver table for sales_personcreditcard not found. Starting initial load.
2026-06-03 22:03:40,052 - INFO - Silver table for sales_salesorderdetail not found. Starting initial load.
2026-06-03 22:03:40,055 - INFO - Silver table for sales_currency not found. Starting initial load.
2026-06-03 22:03:40,059 

## Stop session and clear cash from spark

In [11]:
spark.stop()
spark.catalog.clearCache()